[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter06_gradient_descent_from_scratch/notebook.ipynb)

# Chapter 6: Gradient Descent From Scratch

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

In Chapter 5 you met Carl Friedrich Gauss working out the orbit of Ceres by making the squared errors as small as possible. Forty six years later, another mathematician was stuck on the same kind of problem. Augustin Louis Cauchy, in Paris, was trying to pin down the orbits of planets and comets from telescope observations, and the equations that came out of that work were awkward. The usual approach was to eliminate the unknowns one at a time until a single equation was left, and with several unknowns tangled together that became, in his words, very long work.

On the eighteenth of October 1847, Cauchy presented a short note to the Académie des Sciences with a completely different idea. It was only about three pages long, and it was titled "General method for the resolution of systems of simultaneous equations". Instead of solving the equations directly, he turned them into a single quantity that measured how badly a guess failed, exactly as Gauss had. Then, rather than trying to jump to the answer, he asked a far more modest question. From where I am standing now, which direction is downhill, and how far should I step that way?

Repeat that question enough times and you arrive at the bottom. Cauchy promised a fuller paper with proofs that the method always works, and that paper never appeared. The idea survived anyway, and today it has a name you will use for the rest of this book. It is called **gradient descent**, and it is how essentially every neural network in the world is trained, including the one with 60 million weights that you met in Chapter 1.

In Chapters 2 and 3 you chose a network's weights yourself, by thinking about what they ought to be. This chapter is where that stops. Before you read on, try to guess the answers to these four questions.

1. **If you are standing in thick fog on a hillside, how could you still find your way downwards?**
2. **Once you know which way is downhill, how big a step should you take?**
3. **What would go wrong if your steps were far too large?**
4. **Could a method that only ever walks downhill still end up somewhere that is not the lowest point?**

## The Idea of Rolling Downhill

Imagine you are on a mountainside in fog so thick that you can see only the ground at your feet. You want to reach the valley, and you have no map. You can still make progress, because you can feel which way the ground slopes. You turn until your toes point downhill, take a step, and then feel the ground again from the new spot. You may not take the shortest route, but as long as every step goes downwards you will end up at the bottom.

That is the whole of gradient descent. The hillside is the loss from Chapter 5, your position is the network's weights, and the slope under your feet is what tells you which way to move.

Start with a loss so simple that you can check every number in your head. Suppose the loss for a single weight $w$ is $(w-2)^2$. It is a bowl with its lowest point at $w=2$, where the loss is zero. The **slope** of this bowl at any point is $2(w-2)$, and for now you can take that formula on trust, since the next section explains where a slope comes from.

In [ ]:
def loss(w):
    return (w - 2) ** 2

def slope(w):
    return 2 * (w - 2)

Suppose you start at $w=5$, which is well up the right hand side of the bowl. The slope there is $2(5-2)$, which is 6. A positive slope means the ground rises as $w$ increases, so downhill is the other way, towards smaller $w$. That gives the rule at the heart of everything.

**To go downhill, move the weight in the opposite direction to its slope, by an amount equal to the slope times a small number called the learning rate.**

The **learning rate** is how large a step you take, and here it is 0.3. The first step moves from 5 by $0.3 \times 6 = 1.8$, landing at 3.2. The next cell takes six such steps and prints the slope, the new weight and the new loss each time.

In [ ]:
w = 5.0
for step in range(1, 7):
    g = slope(w)
    w = w - 0.3 * g
    print(step, round(g, 3), round(w, 4), round(loss(w), 4))

The weight slides from 5 towards 2 and the loss falls from 9 to almost nothing. Notice how the steps get smaller on their own. Near the top of the bowl the ground is steep, the slope is large, and the step is big. Near the bottom the ground flattens, the slope shrinks, and the steps become gentle. Nobody arranged that, since it falls out of the rule. Figure 6.1 shows the six steps sitting on the bowl.

> 🖼️ **Figure 6.1:** Six steps of gradient descent down a simple bowl. The steps are large where the bowl is steep and shrink by themselves as the ground flattens out.

## Following the Slope

The slope of a loss with respect to a weight has a proper name, which is the **derivative**, and when a network has many weights the whole collection of slopes is called the **gradient**. The gradient is just a list holding one slope for every weight, saying how steeply the loss rises as each weight increases on its own.

You do not have to take any slope formula on trust, because you can always measure a slope the way a surveyor would. Nudge the weight a tiny bit in each direction, see how much the loss changes, and divide the change in loss by the change in weight. The next cells bring back the café from Chapter 5, which guesses its cold drink sales by multiplying the day's temperature by a single weight.

In [ ]:
import numpy as np

temp = np.array([20, 24, 28, 30, 33, 35])
drinks = np.array([41, 47, 58, 59, 67, 69])

def mse(w):
    return ((w * temp - drinks) ** 2).mean()

Measuring the slope at $w=1.5$ takes two cheap guesses, one just above and one just below.

In [ ]:
h = 0.001
measured = (mse(1.5 + h) - mse(1.5 - h)) / (2 * h)
print(round(measured, 2))

Calculus gives the same answer without the nudging. Differentiating the mean of $(w \times \text{temp} - \text{drinks})^2$ with respect to $w$ gives the mean of $2 \times \text{temp} \times (w \times \text{temp} - \text{drinks})$, and the next cell checks that the formula and the measurement agree.

In [ ]:
def slope_mse(w):
    return (2 * temp * (w * temp - drinks)).mean()

print(round(slope_mse(1.5), 2), round(slope_mse(2.4), 2))

The formula matches the measurement exactly, and the two signs tell the whole story. At $w=1.5$ the slope is negative, meaning the loss falls as $w$ grows, so the weight should increase. At $w=2.4$ the slope is positive, so the weight should decrease. Both point towards the bottom near 2. Figure 6.2 draws the bowl with the ground under your feet at each of those two points.

> 🖼️ **Figure 6.2:** The café's loss bowl with the ground under your feet drawn at two weights. A negative slope means move right, a positive slope means move left, and both lead to the bottom.

**The gradient points in the direction in which the loss rises fastest, so stepping the opposite way is the fastest way down from where you stand.** Measuring slopes by nudging works perfectly well for one weight, and Chapter 7 explains why it becomes hopeless once a network has thousands of them, and what replaces it.

## The Learning Rate, the One Dial That Matters

You now know which way to step. The remaining question is how far, and that is set by the learning rate. It is the single most important setting in deep learning, and the easiest to get wrong, because both too small and too large go badly in completely different ways.

The next cell writes gradient descent as a reusable function. It starts from a weight, takes a given number of steps, and returns every weight it visited along the way.

In [ ]:
def descend(start, learning_rate, steps):
    w = start
    path = [w]
    for _ in range(steps):
        w = w - learning_rate * slope_mse(w)
        path.append(w)
    return path

Now run it three times from the same starting weight of 1.5, with three different learning rates, and print where each one has reached after thirty steps.

In [ ]:
for lr in [0.00005, 0.0003, 0.0013]:
    final = descend(1.5, lr, 30)[-1]
    print(lr, round(final, 4), round(mse(final), 3))

Three learning rates, three completely different outcomes. The smallest one is still crawling and has not arrived after thirty steps. The middle one lands on 2.004, which is exactly the best weight, with the loss of 1.487 you would expect there. The largest one has ended up at $-36$, with a loss of more than a million, which is far worse than where it started.

That last case is worth understanding, because it catches everyone eventually. When the step is too big you overshoot the bottom and land further up the opposite side than you were before. The next step overshoots even harder in the other direction, and the weight bounces between the walls of the bowl, climbing higher each time. The next cell prints the first few weights on that path.

In [ ]:
for step, w in enumerate(descend(1.5, 0.0013, 5)):
    print(step, round(float(w), 3))

The weight jumps over the bottom at 2.0 every single time, and each jump lands further out than the last. Figure 6.3 shows all three paths on the bowl.

> 🖼️ **Figure 6.3:** Three learning rates on the same bowl. Too small a rate crawls, too large a rate bounces off the walls and climbs, and a good rate settles at the bottom.

These learning rates look oddly tiny, and there is a reason worth noticing. The temperatures are numbers around 30, and squaring them in the loss makes the slopes come out in the hundreds, so a step of any useful size needs a very small rate to balance them. Had the data been measured in smaller units, the same bowl would have wanted a much larger rate. **The right learning rate depends on the scale of your data, so there is no single good value to memorise, and finding one is a normal part of training.** Chapter 14 is devoted to finding and adjusting it, and the habit of rescaling inputs so that the numbers are small, which you met in Books 4 and 5, is partly about making this problem easier.

## Watching It Converge

In real training you do not watch the weight, because a network has far too many. You watch the loss. The next cell runs thirty steps at the good learning rate and records the loss at each one.

In [ ]:
path = descend(1.5, 0.0003, 30)
losses = [mse(w) for w in path]
for step in [0, 1, 2, 5, 10, 30]:
    print(step, round(path[step], 4), round(losses[step], 3))

The loss drops steeply at first and then flattens out, which is the shape you will see in almost every training run for the rest of this book. Figure 6.4 plots it.

> 🖼️ **Figure 6.4:** The loss at every step of a run that works. Most of the improvement arrives in the first handful of steps, and the curve then settles onto the lowest loss this model can reach.

A flattening curve is how you know training has **converged**, meaning that further steps are no longer buying much. Notice what it settles on. The loss stops at 1.487 and not at zero, because 1.487 is simply the best this model can do. A straight line through the origin cannot pass exactly through all six days, so some error always remains. **A loss that stops falling means the model has got as far as it can, not that it is perfect, and the number it settles on tells you how good the best possible version of that model is.** When that settled value is too high, the answer is a better model rather than more steps, which in this book usually means more layers or more neurons.

## Why the Hill Is Not Smooth: Local Minima and Saddle Points

Every loss in this chapter has been a single clean bowl, where walking downhill is guaranteed to reach the only bottom there is. Real networks are not like that. With thousands of weights interacting, the loss landscape is more like a mountain range than a bowl, with many separate valleys, long flat plateaus and ridges between them.

A valley that is not the deepest one is called a **local minimum**. Standing at its bottom, every direction leads upwards, so gradient descent stops, even though a far better valley exists somewhere else. The next cells build a small landscape with exactly that shape.

In [ ]:
def hilly(w):
    return w ** 4 - 8 * w ** 2 + 2 * w + 21

def hilly_slope(w):
    return 4 * w ** 3 - 16 * w + 2

The next cell rolls downhill from two starting points that are only two steps apart, at $-1$ and $+1$, taking sixty small steps from each.

In [ ]:
for start in [-1.0, 1.0]:
    w = start
    for _ in range(60):
        w = w - 0.02 * hilly_slope(w)
    print(start, round(w, 4), round(hilly(w), 3))

The two runs end up in different valleys with very different losses. Neither one did anything wrong, since both walked downhill every single step, and the only thing that separated them was where they happened to begin. Figure 6.5 shows the landscape and both journeys.

> 🖼️ **Figure 6.5:** One landscape, two valleys. Both runs walked downhill at every step, and the only difference between them was the starting point.

There is a second kind of place where descent can stall, called a **saddle point**. Picture a mountain pass, or the saddle on a horse's back. Standing in the middle of a pass, the ground falls away down into the valleys on either side, but rises towards the peaks in front of you and behind you. The ground right where you stand is almost flat, so the slope is nearly zero and your steps become tiny, even though a steep descent is available just a short distance away. In networks with many weights, places like this turn out to be far more common than true local minima, because a point is only a genuine minimum if the ground rises in every one of thousands of directions at once, which is a demanding thing to ask.

This all sounds like bad news, and for decades people assumed it would be. In practice, training large networks works far better than that worry suggests, for reasons worth knowing.

- **Most valleys are good enough.** In large networks, the many local minima tend to sit at similar depths, so the one you land in is usually nearly as good as the best.
- **Noise helps.** Chapter 12 trains on small random batches of data rather than all of it at once, which makes every step slightly jittery, and that jitter is often enough to shake a run out of a shallow dip.
- **Momentum helps.** Chapter 13 gives descent a kind of inertia, so it rolls through small dips and across flat ground instead of stopping in them.
- **Starting points matter.** Since where you begin decides where you end, the starting weights are chosen with care rather than at random guesswork, which is the subject of Chapter 16.

**Gradient descent only promises to walk downhill from where it is, not to find the lowest point in the whole landscape, and almost everything in Part 4 of this book exists to make that walk end somewhere good.**

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Start from the other side.** Run `descend(3.0, 0.0003, 30)` and check that it still arrives at 2.004. Look at the sign of `slope_mse(3.0)` and explain which way the first step goes.
2. **Find the breaking point.** Try learning rates of 0.0009, 0.0010 and 0.0011 for thirty steps and find roughly where the runs stop settling and start bouncing.
3. **Climb the hill instead.** Change the minus sign in `descend` to a plus and run it from 1.5. Watch what happens to the loss, and write down in one sentence why the sign matters so much.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **What is the rule for updating a weight in gradient descent?**
2. **How can you measure a slope without doing any calculus?**
3. **Why do the steps get smaller on their own as training goes on?**
4. **What happens when the learning rate is far too large, and why?**
5. **Why can two runs on the same landscape end with different losses?**

### Answers

1. Move the weight opposite to its slope, by the slope times the learning rate.
2. Nudge the weight a tiny amount each way, and divide the change in loss by the change in weight.
3. Because the slope shrinks as the ground flattens near the bottom, and the step is proportional to the slope.
4. The weight overshoots the bottom and lands higher on the far side, then overshoots further each time, so the loss grows.
5. Because the landscape has several valleys, and each run stops in whichever one it walked into from its starting point.

## What Comes Next

You can now find the best weight by walking downhill, measure a slope by hand, recognise a learning rate that is too small or too large, and read a loss curve that has converged.

- **Chapter 7, Backpropagation From the Ground Up,** works out the slope for every weight in a network at once, which is the one piece still missing.
- **Chapter 8, Tensors in PyTorch,** begins the part of the book where a framework does this work for you.

Chapter 7 starts with a problem. Measuring slopes by nudging takes two guesses per weight, and a network with a hundred thousand weights would need two hundred thousand guesses for a single step.